In [1]:
from pathlib import Path

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from statsmodels.nonparametric.kernel_regression import KernelReg
import jax
from jax import jit, random
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from scipy import stats

In [2]:
jax.config.update('jax_enable_x64', True)
collected_vehicle_data_path = Path("..", "raw_data", "collected-data", "collected_vehicle_data.csv")

# Research Question: How frequent are different vehicle body styles in the target population?

As shown in @fig-with-errors, it appars that "SPORT UTILITY 4-DR" and "SEDAN 4-DR" are the clear winners in the sample.  Based on prior experience, the author believes that these body styles are also most frequent in the target population.  

In [3]:
obs = pl.scan_csv(source=collected_vehicle_data_path)

In [4]:
obs_2 = (obs
    .group_by("style_body")
    .agg(pl.len().alias("count"))
    .sort("count", descending=False)
)

n = obs_2.select(pl.col("count").sum()).collect().item()

In [5]:
#############################################################
# Start sensitivity analysis.
#############################################################
@jit
def get_var_binary_data(num_1s, n):
    """
    See: https://stats.stackexchange.com/questions/67019/variance-and-covariance-of-binary-data
    """
    return num_1s * (n - num_1s)/(n * (n - 1))


@jit
def get_phi(n_11, n_00, n_10, n_01):
    """https://en.wikipedia.org/wiki/Phi_coefficient
    
    Matrix arguments are also accepted.
    In this case, the product in the formula in the denominator
    is taken along axis 0.  If matrix arguments are supplied,
    then they must all be of the same shape.  
    """
    n_1_dot = (n_11 + n_10).astype(np.float64)
    n_0_dot = (n_01 + n_00).astype(np.float64)
    n_dot_1 = (n_01 + n_11).astype(np.float64)
    n_dot_0 = (n_00 + n_10).astype(np.float64)

    divisor = jnp.prod(jnp.sqrt(jnp.array([n_1_dot, n_0_dot, n_dot_0, n_dot_1])), axis=0)
    phi = (n_11 * n_00 - n_10 * n_01) / divisor
    phi_2 = jnp.nan_to_num(phi, copy=True)
    return phi_2


def get_num_1s_for_R_0(N, n, alpha_for_R_0):
    probs = stats.dirichlet.rvs(
        alpha=alpha_for_R_0,
        size=1
    )
  
    m = np.ceil((N - n) * probs).astype(np.int64)

    multivariate_hypergeom_obj = stats.multivariate_hypergeom(
        m=m,
        n=N - n
    )

    num_1s_for_R_0 = multivariate_hypergeom_obj.rvs(size=1).ravel()

    return num_1s_for_R_0


@jit
def get_error_of_sample_mean(corr, n, N, pop_std):
    """
    See: Sampling: Design and Analysis by Lohr on p. 528
    """
    return corr * jnp.sqrt((N - 1)/n * (1 - n/N)) * jnp.sqrt(pop_std)


def makes_non_response_model(M:int, N:int, obs_counts:np.ndarray):
    """
    Args:
        M: size of sensitivity analysis 
        obs_counts: sorted by make alphabetically.
            The last entry should be for 'other' makes.

    Returns:
        dict with keys of (
            num_1s_for_R_0_all,
            correlations,
            errors
        )
    """
    num_1s_for_R_1 = obs_counts
    alpha_for_R_0 = obs_counts.copy()
    
    n = np.sum(obs_counts)
    # k is the number of categories.
    k = obs_counts.shape[0]
    num_1s_for_R_0_all = np.empty(shape=(M, k), dtype=np.int64)
    correlations = np.empty(shape=(M, k), dtype=np.float64)
    errors = np.empty(shape=(M, k), dtype=np.float64)

    for m in range(M):
        num_1s_for_R_0 = get_num_1s_for_R_0(
            N=N,
            n=n,
            alpha_for_R_0=alpha_for_R_0
        )

        num_0s_for_R_0 = N - n - num_1s_for_R_0
        num_0s_for_R_1 = n - num_1s_for_R_1

        corr = get_phi(
            n_11=num_1s_for_R_1,
            n_00=num_0s_for_R_0,
            n_10=num_1s_for_R_0,
            n_01=num_0s_for_R_1
        )

        # https://stats.stackexchange.com/questions/67019/variance-and-covariance-of-binary-data
        num_1s = num_1s_for_R_1 + num_1s_for_R_0
        pop_var_for_each_category = get_var_binary_data(num_1s, N)
        pop_std_for_each_category = np.sqrt(pop_var_for_each_category)

        error_for_each_category = get_error_of_sample_mean(
            corr=corr,
            n=n,
            N=N,
            pop_std=pop_std_for_each_category
        )
        

        # Save this loop's results.
        num_1s_for_R_0_all[m, :] = num_1s_for_R_0
        correlations[m, :] = corr
        errors[m, :] = error_for_each_category

    return dict(
        num_1s_for_R_0_all=num_1s_for_R_0_all,
        correlations=correlations,
        errors=errors
    )

In [6]:
obs_counts = obs_2.sort("style_body").select(pl.col("count").cast(pl.Int64)).collect().to_series().to_numpy()
n = np.sum(obs_counts)
obs_props = obs_counts / n
body_styles_series = obs_2.sort("style_body").select("style_body").collect().to_series()
sens_res = makes_non_response_model(
    M=1000,
    N=500000,
    obs_counts=obs_counts
)

In [7]:
absolute_errors = np.abs(sens_res["errors"])
absolute_errors_worst_case = np.quantile(absolute_errors, 0.9, axis=0)

In [8]:
lower = obs_props - absolute_errors_worst_case
lower[lower < 0] = 0
upper = obs_props + absolute_errors_worst_case
upper[upper > 1] = 1

obs_3 = pl.LazyFrame(
    data=dict(
        make=body_styles_series,
        p=obs_props,
        p_lower=lower,
        p_upper=upper
    )
)
obs_3_trans = (obs_3
    .with_columns(
        (pl.col("p") - pl.col("p_lower"))
            .alias("p_lower"),

        (pl.col("p_upper") - pl.col("p"))
            .alias("p_upper")
    )
    .sort("p")                 
)

In [63]:
#| fig-cap: "Solid dots represent observed proportions in the sample.  Error bars go out +/-1 simulated mean absolute error for each body style."
#| label: fig-with-errors
x = body_styles_series
fig = go.Figure(
    data=go.Scatter(
        mode="markers",
        x=obs_3_trans.select("p").collect().to_series(),
        y=obs_3_trans.select("make").collect().to_series(),
        error_x=dict(
            type='data',
            symmetric=False,
            array=obs_3_trans.select("p_upper").collect().to_series(),
            arrayminus=obs_3_trans.select("p_lower").collect().to_series()
        ),
        hoverinfo="text",
        hovertext=obs_2.select((pl.lit("n = ") + pl.col("count").cast(pl.Utf8) + "<br>p = " + (100 * pl.col("count")/n).round(4).cast(pl.Utf8) + "%")).collect().to_series()
    )
)

fig.update_layout(
    title="Vehicle Body Styles",
    xaxis=dict(title="Proportion"),
    yaxis=dict(tickfont_size=11)
)
fig.show()

# Appendix: Sensitivity Analysis

In [51]:
#| fig-cap: Non-response Simulation. The estimate in orange (shown for comparison only) is the result of multiplying the observed proportions by N - n, where N is assumed to be 500000.
#| label: fig-numones
fig = px.line()
rng = np.random.default_rng()
num_1s_for_R_0_all_sample = rng.choice(sens_res["num_1s_for_R_0_all"], size=100, axis=0)
num_traces = num_1s_for_R_0_all_sample.shape[0]
y = body_styles_series
x_all = num_1s_for_R_0_all_sample
for i in range(num_traces):
    x = x_all[i, :]
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            opacity=0.05,
            mode="lines",
            line = dict(color='black'),
            showlegend=False,
            hoverinfo="skip"
        )
    )

x = np.round((500000 - n) * obs_props)
fig.add_trace(
    go.Scatter(
        name=f"Estimate",
        x=x,
        y=y,
        mode="lines",
        line = dict(color="rgb(224, 190, 144)"),
        showlegend=True
    )
)
x = np.quantile(num_1s_for_R_0_all_sample, 0.1, axis=0)
fig.add_trace(
    go.Scatter(
        name=f"0.1-Quantile",
        x=x,
        y=y,
        mode="lines",
        line = dict(color="rgb(140, 210, 219)"),
        showlegend=True
    )
)
x = np.quantile(num_1s_for_R_0_all_sample, 0.9, axis=0)
fig.add_trace(
    go.Scatter(
        name=f"0.9-Quantile",
        x=x,
        y=y,
        mode="lines",
        line = dict(color="rgb(216, 182, 226)"),
        showlegend=True
    )
)
fig.update_layout(
    title=f"Simulation of Numbers of Vehicles of Each Body Style among Non-Responders",
    yaxis={"title": "Body Style"},
    xaxis={"title": "Frequency"},
    hovermode="y unified"
)

fig.show()

In [65]:
fig = px.line()
rng = np.random.default_rng()
x_all = rng.choice(sens_res["correlations"], size=100, axis=0)
num_traces = x_all.shape[0]
y = body_styles_series

for i in range(num_traces):
    x = x_all[i, :]
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            opacity=0.05,
            mode="lines",
            line = dict(color='black'),
            showlegend=False
        )
    )


fig.update_layout(
    title=f"Simulation of Correlations Between Responders & Non-Responders",
    yaxis={"title": "Body Style"},
    xaxis={"title": "Correlation"}
)

fig.show()

In [68]:
fig = px.line()
rng = np.random.default_rng()
x_all = rng.choice(absolute_errors, size=100, axis=0)
num_traces = x_all.shape[0]
y = body_styles_series

for i in range(num_traces):
    x = x_all[i, :]
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            opacity=0.05,
            mode="lines",
            line = dict(color='black'),
            showlegend=False
        )
    )
fig.add_trace(
        go.Scatter(
            name="0.9-quantile",
            x=absolute_errors_worst_case,
            y=y,
            mode="lines",
            line = dict(color='rgb(199, 21, 232)'),
            showlegend=True
        )
    )

fig.update_layout(
    title=f"Simulation of Absolute Errors<br>in Relative Frequencies of Different Body Styles",
    yaxis={"title": "Body Style"},
    xaxis={"title": "Absolute Error"}
)

fig.show()